In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import re

# 🔽 Ваши импорты
from readerscf import parse_sdr_file
from get_project_root import get_project_root
from substractdf import subtract_column_min
from center_dataframe import center_dataframe
from estimate_M_from_data import estimate_M_from_data
from estimate_crosstalk_matrix import estimate_crosstalk_matrix
from divide_matrices_np import divide_matrices_np

def sanitize_sheet_name(filename: str) -> str:
    """Приводит имя файла к формату, допустимому для листа Excel (макс 31 символ, без спецсимволов)."""
    name = Path(filename).stem  # убираем .srd
    name = re.sub(r'[\\\/\?\*\[\]:]', '', name)
    if len(name) > 31:
        name = name[:28] + "..."
    return name

def save_matrices_to_excel(results: list, output_path: Path):
    """Сохраняет 4 матрицы для каждого файла в отдельные листы одного Excel-файла."""
    if not results:
        print("⚠️ Нет данных для сохранения.")
        return

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        used_names = {}
        
        for res in results:
            # Уникальное имя листа
            base_name = sanitize_sheet_name(res['file'])
            if base_name in used_names:
                used_names[base_name] += 1
                sheet_name = f"{base_name}_{used_names[base_name]}"
            else:
                used_names[base_name] = 0
                sheet_name = base_name

            start_row = 0
            matrices = [
                ('matrixdef',    '🔹 M0: Default'),
                ('matrix1',    '🔹 M1: Estimated'),
                ('matrix2',    '🔹 M2: Crosstalk'),
                ('matrix1def', '🔹 M1: Normalized'),
                ('matrix2def', '🔹 M2: Normalized')
            ]

            for mat_key, title in matrices:
                df = pd.DataFrame(res[mat_key])
                df.index = [f'Row {i+1}' for i in range(df.shape[0])]
                df.columns = [f'Col {j+1}' for j in range(df.shape[1])]
                
                # Заголовок матрицы
                pd.DataFrame([[title]]).to_excel(writer, sheet_name=sheet_name, 
                                                 startrow=start_row, index=False, header=False)
                # Сама матрица (через 1 строку после заголовка)
                df.to_excel(writer, sheet_name=sheet_name, 
                            startrow=start_row + 2, index=True, header=True)
                
                start_row += df.shape[0] + 4  # данные + заголовок + отступ + заголовок следующей

            tqdm.write(f"   📄 Лист '{sheet_name}' сохранён")

    print(f"💾 Итоговый файл: {output_path}")


def main():
    project_root = Path(get_project_root())
    print(f"📁 Корень проекта: {project_root}")

    srd_dir = project_root / "files" / "сырые SRD" / "сырые SRD"
    if not srd_dir.is_dir():
        print(f"❌ Папка не найдена: {srd_dir}")
        return

    srd_files = sorted(srd_dir.glob("*.srd"))
    if not srd_files:
        print("⚠️ Файлы .srd не найдены.")
        return

    print(f"📂 Найдено файлов: {len(srd_files)}\n")

    results = []  # 📦 Накопитель результатов

    with tqdm(srd_files, desc="🔄 Обработка SRD", unit="файл") as pbar:
        for srd_path in pbar:
            pbar.set_postfix(file=srd_path.name[:18] + "..." if len(srd_path.name) > 18 else srd_path.name)
            
            try:
                sdr_matrix, sdr_channels, sdr_meta =parse_sdr_file(srd_path)
                matrix = sdr_matrix.to_numpy()
                
                if matrix.shape[0] < 10 or matrix.shape[1] < 4:
                    tqdm.write(f"   ⚠️ Матрица слишком мала, пропуск.")
                    continue
                    
                matrixdef = matrix[6:10, 0:4]
                
                data0 = sdr_channels.loc[:, ['dR110', 'dR6G', 'dTAMRA', 'dROX']].copy()
                data0.columns = ['G', 'A', 'T', 'C']
                data0 = subtract_column_min(data0)
                data0 = center_dataframe(data0, method='percentile', percentile=2)
                
                matrix1 = estimate_M_from_data(
                    raw=data0, dye_order=['G', 'A', 'T', 'C'],
                    min_purity=0.8, peak_height=400,
                    peak_distance=20, peak_prominence=150
                )
                matrix2 = estimate_crosstalk_matrix(data0, init_M=matrixdef)
                
                matrix1def = divide_matrices_np(matrix1, matrixdef)
                matrix2def = divide_matrices_np(matrix2, matrixdef)

                # 📥 Сохраняем в память
                results.append({
                    'file': srd_path.name,
                    'matrixdef': matrixdef, 
                    'matrix1': matrix1, 'matrix2': matrix2,
                    'matrix1def': matrix1def, 'matrix2def': matrix2def
                })
                
            except Exception as e:
                tqdm.write(f"   ❌ Ошибка в {srd_path.name}: {e}")

    # 💾 Запись в Excel (папка result)
    output_dir = project_root / "results"/"excel"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "srd_all_matrices.xlsx"
    save_matrices_to_excel(results, output_file)


if __name__ == "__main__":
    main()

📁 Корень проекта: C:\Users\Admin\Documents\GitHub\dnasegnercrosstalk
📂 Найдено файлов: 40



🔄 Обработка SRD:   2%|▉                                  | 1/40 [00:00<00:11,  3.29файл/s, file=3_pGEM_A3_PDMA6_50...]

Предупреждение: для канала 0 (G) не найдено чистых пиков. Использован единичный вектор как fallback.
Предупреждение: для канала 3 (C) не найдено чистых пиков. Использован единичный вектор как fallback.
Найдено пиков: 592
  Итерация 1: max Δ = 0.625512
  Итерация 2: max Δ = 0.026133
  Итерация 3: max Δ = 0.010547
  Сходимость на итерации 5


🔄 Обработка SRD:   5%|█▊                                 | 2/40 [00:00<00:19,  1.92файл/s, file=3_pGEM_B3_PDMA6_50...]

Предупреждение: для канала 0 (G) не найдено чистых пиков. Использован единичный вектор как fallback.
Предупреждение: для канала 1 (A) не найдено чистых пиков. Использован единичный вектор как fallback.
Предупреждение: для канала 2 (T) не найдено чистых пиков. Использован единичный вектор как fallback.
Предупреждение: для канала 3 (C) не найдено чистых пиков. Использован единичный вектор как fallback.
Найдено пиков: 871
  Итерация 1: max Δ = 0.619864
  Итерация 2: max Δ = 0.010523
  Итерация 3: max Δ = 0.002804
  Сходимость на итерации 5


...

🔄 Обработка SRD:  55%|██████████████████▋               | 22/40 [00:15<00:08,  2.21файл/s, file=6_pGEM_2_D3_PDMA6_...]

Предупреждение: для канала 0 (G) не найдено чистых пиков. Использован единичный вектор как fallback.
Предупреждение: для канала 1 (A) не найдено чистых пиков. Использован единичный вектор как fallback.
Предупреждение: для канала 3 (C) не найдено чистых пиков. Использован единичный вектор как fallback.
Найдено пиков: 586
  Итерация 1: max Δ = 0.615424
  Итерация 2: max Δ = 0.009628
  Итерация 3: max Δ = 0.003209
  Сходимость на итерации 5


🔄 Обработка SRD:  57%|███████████████████▌              | 23/40 [00:15<00:07,  2.35файл/s, file=6_pGEM_3_E3_PDMA6_...]

Предупреждение: для канала 0 (G) не найдено чистых пиков. Использован единичный вектор как fallback.
Предупреждение: для канала 1 (A) не найдено чистых пиков. Использован единичный вектор как fallback.
Предупреждение: для канала 3 (C) не найдено чистых пиков. Использован единичный вектор как fallback.
Найдено пиков: 593
  Итерация 1: max Δ = 0.600783
  Итерация 2: max Δ = 0.012725
  Итерация 3: max Δ = 0.001558
  Сходимость на итерации 4


🔄 Обработка SRD:  60%|████████████████████▍             | 24/40 [00:16<00:06,  2.60файл/s, file=6_pGEM_3_F3_PDMA6_...]